In [13]:
import sys
import tensorflow as tf

print("Python-Version:", sys.version)
print("GPU verfügbar:", tf.config.list_physical_devices("GPU"))

Python-Version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
GPU verfügbar: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [ ]:
import shutil
import os

src_dir = '/kaggle/input/datasets/ayoubvn/ppgtoecg/ppg-to-ecg-deep-learning'
dst_dir = '/kaggle/working/ppg-to-ecg-deep-learning'

if not os.path.exists(dst_dir):
    shutil.copytree(src_dir, dst_dir)
    print("Projekt erfolgreich nach /kaggle/working kopiert!")
else:
    print("Projekt existiert bereits in /kaggle/working.")



#  Abhängigkeiten installieren
!pip install -r requirements.txt

Projekt erfolgreich nach /kaggle/working kopiert!
Wir arbeiten jetzt in: /kaggle/working/ppg-to-ecg-deep-learning


In [11]:
import os

# Pfad zum src-Ordner auf Kaggle
src_dir = '/kaggle/working/ppg-to-ecg-deep-learning/src'
os.makedirs(src_dir, exist_ok=True)

# Korrigierter Code für architecture.py
architecture_code = '''import tensorflow as tf
from tensorflow.keras import layers, models

def build_model_1_ppg_only(window_size=300):
    inputs = layers.Input(shape=(window_size, 1), name="ppg_input")
    
    # Encoder
    x = layers.Conv1D(32, kernel_size=5, activation="relu", padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(64, kernel_size=5, activation="relu", padding="same")(x)
    
    # Decoder / Reconstruction
    x = layers.Conv1D(64, kernel_size=5, activation="relu", padding="same")(x)
    x = layers.Conv1D(32, kernel_size=5, activation="relu", padding="same")(x)
    
    # Mit padding="same" bleibt die Länge exakt bei 300
    outputs = layers.Conv1D(1, kernel_size=5, activation="linear", padding="same", name="ecg_output")(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="Model_1_PPG_Only")
    return model

def build_model_2_multimodal(window_size=300):
    inputs = layers.Input(shape=(window_size, 2), name="multimodal_input")
    
    # Encoder with multi-channel features
    x = layers.Conv1D(32, kernel_size=5, activation="relu", padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(64, kernel_size=5, activation="relu", padding="same")(x)
    
    # Decoder / Reconstruction
    x = layers.Conv1D(64, kernel_size=5, activation="relu", padding="same")(x)
    x = layers.Conv1D(32, kernel_size=5, activation="relu", padding="same")(x)
    
    outputs = layers.Conv1D(1, kernel_size=5, activation="linear", padding="same", name="clean_ecg_output")(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="Model_2_MultiModal")
    return model
'''

# Datei direkt auf Kaggle schreiben
with open(os.path.join(src_dir, 'architecture.py'), 'w') as f:
    f.write(architecture_code)

print("SUCCESS: architecture.py wurde direkt auf dem Kaggle-Server aktualisiert!")

# Direkt danach das Training starten
os.chdir('/kaggle/working/ppg-to-ecg-deep-learning')
!python src/train.py

SUCCESS: architecture.py wurde direkt auf dem Kaggle-Server aktualisiert!
--- SCHRITT 1: Datensatz laden und vorverarbeiten ---
[SUCCESS] Found 53 valid signal files! Inspecting columns...
[DEBUG] Columns found in bidmc_43_Signals.csv:
['Time [s]', ' RESP', ' PLETH', ' II', ' V', ' AVR']
[COMPLETE] Successfully processed 10600 total window samples!
Training Set: 8480 Samples | Test Set: 2120 Samples

--- SCHRITT 2: Modell 1 trainieren (PPG -> ECG) ---
I0000 00:00:1793126759.540877     708 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1793126759.542922     708 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
Epoch 1/5
I0000 00:00:1793126765.490806     740 device_compiler.h:196] Compiled cluster using XLA!  Th

In [12]:
import os

# Pfad zum src-Ordner auf Kaggle
src_dir = '/kaggle/working/ppg-to-ecg-deep-learning/src'
os.makedirs(src_dir, exist_ok=True)

# Saubere evaluate.py ohne fehlerhafte LaTeX-Dollarzeichen
evaluate_code = '''import matplotlib.pyplot as plt
import numpy as np

def generate_comparison_image(true_ecg, pred_m1, pred_m2, sample_idx=0, filename="outputs/comparison.png"):
    plt.figure(figsize=(12, 6))
    
    plt.plot(true_ecg, label="True ECG", color="black", linewidth=2)
    plt.plot(pred_m1, label="Model 1 (PPG -> ECG)", color="blue", linestyle="--")
    plt.plot(pred_m2, label="Model 2 (Multi-Modal -> Clean ECG)", color="red", linestyle="-.")
    
    plt.title(f"ECG Reconstruction Comparison (Sample {sample_idx})")
    plt.xlabel("Time Steps")
    plt.ylabel("Amplitude")
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()
    print(f"[SUCCESS] Comparison image saved to {filename}")
'''

# Datei direkt auf Kaggle schreiben
with open(os.path.join(src_dir, 'evaluate.py'), 'w') as f:
    f.write(evaluate_code)

print("SUCCESS: evaluate.py wurde erfolgreich korrigiert und aktualisiert!")

# Training und Evaluation erneut starten
os.chdir('/kaggle/working/ppg-to-ecg-deep-learning')
!python src/train.py

SUCCESS: evaluate.py wurde erfolgreich korrigiert und aktualisiert!
--- SCHRITT 1: Datensatz laden und vorverarbeiten ---
[SUCCESS] Found 53 valid signal files! Inspecting columns...
[DEBUG] Columns found in bidmc_43_Signals.csv:
['Time [s]', ' RESP', ' PLETH', ' II', ' V', ' AVR']
[COMPLETE] Successfully processed 10600 total window samples!
Training Set: 8480 Samples | Test Set: 2120 Samples

--- SCHRITT 2: Modell 1 trainieren (PPG -> ECG) ---
I0000 00:00:1793126994.600299    1084 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1793126994.602339    1084 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
Epoch 1/5
I0000 00:00:1793126998.916941    1114 device_compiler.h:196] Compiled cluster using XLA!  This lin

In [13]:
!pip install optuna

In [14]:
import os

src_dir = '/kaggle/working/ppg-to-ecg-deep-learning/src'
os.makedirs(src_dir, exist_ok=True)

optimize_code = '''import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import optuna
from dataset import load_and_preprocess_data

def objective(trial):
    # Hyperparameter definieren, die Optuna optimieren soll
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    filters_1 = trial.suggest_categorical("filters_1", [16, 32, 64])
    filters_2 = trial.suggest_categorical("filters_2", [32, 64, 128])
    
    # Daten laden
    X_ppg, _, Y_ecg = load_and_preprocess_data(data_dir="data", window_size=300)
    split = int(0.8 * X_ppg.shape[0])
    X_train, X_val = X_ppg[:split], X_ppg[split:]
    y_train, y_val = Y_ecg[:split], Y_ecg[split:]
    
    # Modell mit den Parametern des Trials aufbauen
    inputs = layers.Input(shape=(300, 1), name="ppg_input")
    x = layers.Conv1D(filters_1, kernel_size=5, activation="relu", padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(filters_2, kernel_size=5, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters_2, kernel_size=5, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters_1, kernel_size=5, activation="relu", padding="same")(x)
    outputs = layers.Conv1D(1, kernel_size=5, activation="linear", padding="same", name="ecg_output")(x)
    
    model = models.Model(inputs=inputs, outputs=outputs)
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="mse",
        metrics=["mae"]
    )
    
    # Kurzes Training (z. B. 5 Epochen pro Trial, um Zeit zu sparen)
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=5,
        batch_size=32,
        verbose=0
    )
    
    # Wir optimieren auf minimalen Validation-Loss
    val_loss = min(history.history["val_loss"])
    return val_loss

def main():
    print("--- STARTE OPTUNA HYPERPARAMETER-OPTIMIERUNG ---")
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=10) # 10 Durchläufe (kann angepasst werden)
    
    print("\\n[ERGEBNISSE] Beste Hyperparameter gefunden:")
    print(study.best_params)
    print(f"Bester Validation-Loss: {study.best_value:.4f}")

if __name__ == "__main__":
    main()
'''

# Skript auf dem Server speichern
with open(os.path.join(src_dir, 'optimize.py'), 'w') as f:
    f.write(optimize_code)

print("SUCCESS: optimize.py wurde erstellt!")

# In den Projektordner wechseln und die Optimierung starten
os.chdir('/kaggle/working/ppg-to-ecg-deep-learning')
!python src/optimize.py

SUCCESS: optimize.py wurde erstellt!
--- STARTE OPTUNA HYPERPARAMETER-OPTIMIERUNG ---
[I 2026-10-27 18:55:18,739] A new study created in memory with name: no-name-8265f07a-6c6c-49a8-9463-3e7194619853
[SUCCESS] Found 53 valid signal files! Inspecting columns...
[DEBUG] Columns found in bidmc_43_Signals.csv:
['Time [s]', ' RESP', ' PLETH', ' II', ' V', ' AVR']
[COMPLETE] Successfully processed 10600 total window samples!
I0000 00:00:1793127321.754514    1483 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1793127321.756542    1483 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1793127326.102662    1515 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the life

In [15]:
import os

src_dir = '/kaggle/working/ppg-to-ecg-deep-learning/src'
os.makedirs(src_dir, exist_ok=True)

# 1. Architektur mit den optimierten Filtern aktualisieren
architecture_code = '''import tensorflow as tf
from tensorflow.keras import layers, models

def build_model_1_ppg_only(window_size=300):
    inputs = layers.Input(shape=(window_size, 1), name="ppg_input")
    
    # Encoder (mit Optuna-Bestwerten: 64 -> 128 Filter)
    x = layers.Conv1D(64, kernel_size=5, activation="relu", padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(128, kernel_size=5, activation="relu", padding="same")(x)
    
    # Decoder / Reconstruction
    x = layers.Conv1D(128, kernel_size=5, activation="relu", padding="same")(x)
    x = layers.Conv1D(64, kernel_size=5, activation="relu", padding="same")(x)
    
    outputs = layers.Conv1D(1, kernel_size=5, activation="linear", padding="same", name="ecg_output")(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="Model_1_PPG_Only")
    return model

def build_model_2_multimodal(window_size=300):
    inputs = layers.Input(shape=(window_size, 2), name="multimodal_input")
    
    # Encoder with multi-channel features
    x = layers.Conv1D(64, kernel_size=5, activation="relu", padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(128, kernel_size=5, activation="relu", padding="same")(x)
    
    # Decoder / Reconstruction
    x = layers.Conv1D(128, kernel_size=5, activation="relu", padding="same")(x)
    x = layers.Conv1D(64, kernel_size=5, activation="relu", padding="same")(x)
    
    outputs = layers.Conv1D(1, kernel_size=5, activation="linear", padding="same", name="clean_ecg_output")(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="Model_2_MultiModal")
    return model
'''

with open(os.path.join(src_dir, 'architecture.py'), 'w') as f:
    f.write(architecture_code)


# 2. Training mit 50 Epochen und der optimierten Lernrate (0.00118) aktualisieren
train_code = '''import os
import numpy as np
import tensorflow as tf
from architecture import build_model_1_ppg_only, build_model_2_multimodal
from dataset import load_and_preprocess_data
from evaluate import generate_comparison_image

def main():
    print("--- SCHRITT 1: Datensatz laden und vorverarbeiten ---")
    X_ppg, X_noisy_ecg, Y_ecg = load_and_preprocess_data(data_dir="data", window_size=300)
    
    num_samples = X_ppg.shape[0]
    if num_samples == 0:
        print("[FEHLER] Keine Daten geladen.")
        return
        
    split = int(0.8 * num_samples)
    
    X_train_m1, X_test_m1 = X_ppg[:split], X_ppg[split:]
    X_multimodal = np.concatenate([X_ppg, X_noisy_ecg], axis=-1)
    X_train_m2, X_test_m2 = X_multimodal[:split], X_multimodal[split:]
    y_train, y_test = Y_ecg[:split], Y_ecg[split:]
    
    print(f"Training Set: {split} Samples | Test Set: {num_samples - split} Samples")
    
    optimal_lr = 0.0011748264222019513
    
    print("\\n--- SCHRITT 2: Modell 1 trainieren (Optimiertes Training, 50 Epochen) ---")
    model_1 = build_model_1_ppg_only(window_size=300)
    model_1.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=optimal_lr), loss="mse", metrics=["mae"])
    model_1.fit(X_train_m1, y_train, epochs=50, batch_size=32, validation_split=0.2)
    
    print("\\n--- SCHRITT 3: Modell 2 trainieren (Optimiertes Training, 50 Epochen) ---")
    model_2 = build_model_2_multimodal(window_size=300)
    model_2.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=optimal_lr), loss="mse", metrics=["mae"])
    model_2.fit(X_train_m2, y_train, epochs=50, batch_size=32, validation_split=0.2)
    
    print("\\n--- SCHRITT 4: Evaluation & Generierung des finalen Vergleichsbildes ---")
    os.makedirs("outputs", exist_ok=True)
    
    test_idx = 0
    pred_m1 = model_1.predict(X_test_m1[test_idx : test_idx + 1])[0]
    pred_m2 = model_2.predict(X_test_m2[test_idx : test_idx + 1])[0]
    true_sample = y_test[test_idx]
    
    output_filename = "outputs/final_ecg_model_comparison.png"
    generate_comparison_image(
        true_ecg=true_sample.flatten(),
        pred_m1=pred_m1.flatten(),
        pred_m2=pred_m2.flatten(),
        sample_idx=test_idx,
        filename=output_filename
    )
    
    print(f"[FERTIG] Optimiertes Training beendet! Ergebnis gespeichert unter '{output_filename}'.")

if __name__ == "__main__":
    main()
'''

with open(os.path.join(src_dir, 'train.py'), 'w') as f:
    f.write(train_code)

print("SUCCESS: Architektur und Trainingsskript wurden mit den Optuna-Parametern aktualisiert!")

# In den Projektordner wechseln und das finale Training starten
os.chdir('/kaggle/working/ppg-to-ecg-deep-learning')
!python src/train.py

SUCCESS: Architektur und Trainingsskript wurden mit den Optuna-Parametern aktualisiert!
--- SCHRITT 1: Datensatz laden und vorverarbeiten ---
[SUCCESS] Found 53 valid signal files! Inspecting columns...
[DEBUG] Columns found in bidmc_43_Signals.csv:
['Time [s]', ' RESP', ' PLETH', ' II', ' V', ' AVR']
[COMPLETE] Successfully processed 10600 total window samples!
Training Set: 8480 Samples | Test Set: 2120 Samples

--- SCHRITT 2: Modell 1 trainieren (Optimiertes Training, 50 Epochen) ---
I0000 00:00:1793127587.296257    2959 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1793127587.298560    2959 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
Epoch 1/50
I0000 00:00:1793127591.679854    2991 device_compiler.h

In [16]:
import os

src_dir = '/kaggle/working/ppg-to-ecg-deep-learning/src'
os.makedirs(src_dir, exist_ok=True)

# 1. evaluate.py erweitern (plottet Loss & MAE über die Epochen)
evaluate_code = '''import matplotlib.pyplot as plt
import numpy as np

def generate_comparison_image(true_ecg, pred_m1, pred_m2, sample_idx=0, filename="outputs/comparison.png"):
    plt.figure(figsize=(12, 6))
    plt.plot(true_ecg, label="True ECG", color="black", linewidth=2)
    plt.plot(pred_m1, label="Model 1 (PPG -> ECG)", color="blue", linestyle="--")
    plt.plot(pred_m2, label="Model 2 (Multi-Modal -> Clean ECG)", color="red", linestyle="-.")
    plt.title(f"ECG Reconstruction Comparison (Sample {sample_idx})")
    plt.xlabel("Time Steps")
    plt.ylabel("Amplitude")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()
    print(f"[SUCCESS] Comparison image saved to {filename}")

def plot_training_history(history_m1, history_m2, filename="outputs/training_history.png"):
    plt.figure(figsize=(14, 5))
    
    # Loss (MSE) Subplot
    plt.subplot(1, 2, 1)
    plt.plot(history_m1.history['loss'], label='Model 1 Train Loss', color='blue')
    plt.plot(history_m1.history['val_loss'], label='Model 1 Val Loss', color='blue', linestyle='--')
    plt.plot(history_m2.history['loss'], label='Model 2 Train Loss', color='red')
    plt.plot(history_m2.history['val_loss'], label='Model 2 Val Loss', color='red', linestyle='--')
    plt.title("Model Loss (MSE) over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Loss (MSE)")
    plt.legend()
    plt.grid(True)
    
    # MAE Subplot
    plt.subplot(1, 2, 2)
    plt.plot(history_m1.history['mae'], label='Model 1 Train MAE', color='blue')
    plt.plot(history_m1.history['val_mae'], label='Model 1 Val MAE', color='blue', linestyle='--')
    plt.plot(history_m2.history['mae'], label='Model 2 Train MAE', color='red')
    plt.plot(history_m2.history['val_mae'], label='Model 2 Val MAE', color='red', linestyle='--')
    plt.title("Model MAE over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Mean Absolute Error (MAE)")
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()
    print(f"[SUCCESS] Training history plot saved to {filename}")
'''

with open(os.path.join(src_dir, 'evaluate.py'), 'w') as f:
    f.write(evaluate_code)


# 2. train.py aktualisieren, damit die Historie erfasst und geplottet wird
train_code = '''import os
import numpy as np
import tensorflow as tf
from architecture import build_model_1_ppg_only, build_model_2_multimodal
from dataset import load_and_preprocess_data
from evaluate import generate_comparison_image, plot_training_history

def main():
    print("--- SCHRITT 1: Datensatz laden und vorverarbeiten ---")
    X_ppg, X_noisy_ecg, Y_ecg = load_and_preprocess_data(data_dir="data", window_size=300)
    
    num_samples = X_ppg.shape[0]
    if num_samples == 0:
        print("[FEHLER] Keine Daten geladen.")
        return
        
    split = int(0.8 * num_samples)
    
    X_train_m1, X_test_m1 = X_ppg[:split], X_ppg[split:]
    X_multimodal = np.concatenate([X_ppg, X_noisy_ecg], axis=-1)
    X_train_m2, X_test_m2 = X_multimodal[:split], X_multimodal[split:]
    y_train, y_test = Y_ecg[:split], Y_ecg[split:]
    
    print(f"Training Set: {split} Samples | Test Set: {num_samples - split} Samples")
    
    optimal_lr = 0.0011748264222019513
    
    print("\\n--- SCHRITT 2: Modell 1 trainieren (50 Epochen) ---")
    model_1 = build_model_1_ppg_only(window_size=300)
    model_1.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=optimal_lr), loss="mse", metrics=["mae"])
    history_m1 = model_1.fit(X_train_m1, y_train, epochs=50, batch_size=32, validation_split=0.2)
    
    print("\\n--- SCHRITT 3: Modell 2 trainieren (50 Epochen) ---")
    model_2 = build_model_2_multimodal(window_size=300)
    model_2.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=optimal_lr), loss="mse", metrics=["mae"])
    history_m2 = model_2.fit(X_train_m2, y_train, epochs=50, batch_size=32, validation_split=0.2)
    
    print("\\n--- SCHRITT 4: Evaluation & Diagramme generieren ---")
    os.makedirs("outputs", exist_ok=True)
    
    # Trainingsverlauf (Loss & MAE) plotten
    plot_training_history(history_m1, history_m2, filename="outputs/training_history.png")
    
    # Test-Sample Vorhersagevergleich plotten
    test_idx = 0
    pred_m1 = model_1.predict(X_test_m1[test_idx : test_idx + 1])[0]
    pred_m2 = model_2.predict(X_test_m2[test_idx : test_idx + 1])[0]
    true_sample = y_test[test_idx]
    
    generate_comparison_image(
        true_ecg=true_sample.flatten(),
        pred_m1=pred_m1.flatten(),
        pred_m2=pred_m2.flatten(),
        sample_idx=test_idx,
        filename="outputs/final_ecg_model_comparison.png"
    )
    
    print("[FERTIG] Alle Trainings- und Auswertungsplots wurden erfolgreich in 'outputs/' gespeichert!")

if __name__ == "__main__":
    main()
'''

with open(os.path.join(src_dir, 'train.py'), 'w') as f:
    f.write(train_code)

print("SUCCESS: Skripte wurden aktualisiert. Starte das Training mit Genauigkeits-Verlauf...")

# Training ausführen
os.chdir('/kaggle/working/ppg-to-ecg-deep-learning')
!python src/train.py

SUCCESS: Skripte wurden aktualisiert. Starte das Training mit Genauigkeits-Verlauf...
--- SCHRITT 1: Datensatz laden und vorverarbeiten ---
[SUCCESS] Found 53 valid signal files! Inspecting columns...
[DEBUG] Columns found in bidmc_43_Signals.csv:
['Time [s]', ' RESP', ' PLETH', ' II', ' V', ' AVR']
[COMPLETE] Successfully processed 10600 total window samples!
Training Set: 8480 Samples | Test Set: 2120 Samples

--- SCHRITT 2: Modell 1 trainieren (50 Epochen) ---
I0000 00:00:1793128183.531939    5520 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1793128183.533928    5520 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
Epoch 1/50
I0000 00:00:1793128187.964000    5550 device_compiler.h:196] Compiled cluster u

In [17]:
import os

src_dir = '/kaggle/working/ppg-to-ecg-deep-learning/src'
os.makedirs(src_dir, exist_ok=True)

# 1. evaluate.py erweitern (plottet Loss & MAE über die Epochen)
evaluate_code = '''import matplotlib.pyplot as plt
import numpy as np

def generate_comparison_image(true_ecg, pred_m1, pred_m2, sample_idx=0, filename="outputs/comparison.png"):
    plt.figure(figsize=(12, 6))
    plt.plot(true_ecg, label="True ECG", color="black", linewidth=2)
    plt.plot(pred_m1, label="Model 1 (PPG -> ECG)", color="blue", linestyle="--")
    plt.plot(pred_m2, label="Model 2 (Multi-Modal -> Clean ECG)", color="red", linestyle="-.")
    plt.title(f"ECG Reconstruction Comparison (Sample {sample_idx})")
    plt.xlabel("Time Steps")
    plt.ylabel("Amplitude")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()
    print(f"[SUCCESS] Comparison image saved to {filename}")

def plot_training_history(history_m1, history_m2, filename="outputs/training_history.png"):
    plt.figure(figsize=(14, 5))
    
    # Loss (MSE) Subplot
    plt.subplot(1, 2, 1)
    plt.plot(history_m1.history['loss'], label='Model 1 Train Loss', color='blue')
    plt.plot(history_m1.history['val_loss'], label='Model 1 Val Loss', color='blue', linestyle='--')
    plt.plot(history_m2.history['loss'], label='Model 2 Train Loss', color='red')
    plt.plot(history_m2.history['val_loss'], label='Model 2 Val Loss', color='red', linestyle='--')
    plt.title("Model Loss (MSE) over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Loss (MSE)")
    plt.legend()
    plt.grid(True)
    
    # MAE Subplot
    plt.subplot(1, 2, 2)
    plt.plot(history_m1.history['mae'], label='Model 1 Train MAE', color='blue')
    plt.plot(history_m1.history['val_mae'], label='Model 1 Val MAE', color='blue', linestyle='--')
    plt.plot(history_m2.history['mae'], label='Model 2 Train MAE', color='red')
    plt.plot(history_m2.history['val_mae'], label='Model 2 Val MAE', color='red', linestyle='--')
    plt.title("Model MAE over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Mean Absolute Error (MAE)")
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()
    print(f"[SUCCESS] Training history plot saved to {filename}")
'''

with open(os.path.join(src_dir, 'evaluate.py'), 'w') as f:
    f.write(evaluate_code)


# 2. train.py aktualisieren, damit die Historie erfasst und geplottet wird
train_code = '''import os
import numpy as np
import tensorflow as tf
from architecture import build_model_1_ppg_only, build_model_2_multimodal
from dataset import load_and_preprocess_data
from evaluate import generate_comparison_image, plot_training_history

def main():
    print("--- SCHRITT 1: Datensatz laden und vorverarbeiten ---")
    X_ppg, X_noisy_ecg, Y_ecg = load_and_preprocess_data(data_dir="data", window_size=300)
    
    num_samples = X_ppg.shape[0]
    if num_samples == 0:
        print("[FEHLER] Keine Daten geladen.")
        return
        
    split = int(0.8 * num_samples)
    
    X_train_m1, X_test_m1 = X_ppg[:split], X_ppg[split:]
    X_multimodal = np.concatenate([X_ppg, X_noisy_ecg], axis=-1)
    X_train_m2, X_test_m2 = X_multimodal[:split], X_multimodal[split:]
    y_train, y_test = Y_ecg[:split], Y_ecg[split:]
    
    print(f"Training Set: {split} Samples | Test Set: {num_samples - split} Samples")
    
    optimal_lr = 0.0011748264222019513
    
    print("\\n--- SCHRITT 2: Modell 1 trainieren (50 Epochen) ---")
    model_1 = build_model_1_ppg_only(window_size=300)
    model_1.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=optimal_lr), loss="mse", metrics=["mae"])
    history_m1 = model_1.fit(X_train_m1, y_train, epochs=50, batch_size=32, validation_split=0.2)
    
    print("\\n--- SCHRITT 3: Modell 2 trainieren (50 Epochen) ---")
    model_2 = build_model_2_multimodal(window_size=300)
    model_2.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=optimal_lr), loss="mse", metrics=["mae"])
    history_m2 = model_2.fit(X_train_m2, y_train, epochs=50, batch_size=32, validation_split=0.2)
    
    print("\\n--- SCHRITT 4: Evaluation & Diagramme generieren ---")
    os.makedirs("outputs", exist_ok=True)
    
    # Trainingsverlauf (Loss & MAE) plotten
    plot_training_history(history_m1, history_m2, filename="outputs/training_history.png")
    
    # Test-Sample Vorhersagevergleich plotten
    test_idx = 0
    pred_m1 = model_1.predict(X_test_m1[test_idx : test_idx + 1])[0]
    pred_m2 = model_2.predict(X_test_m2[test_idx : test_idx + 1])[0]
    true_sample = y_test[test_idx]
    
    generate_comparison_image(
        true_ecg=true_sample.flatten(),
        pred_m1=pred_m1.flatten(),
        pred_m2=pred_m2.flatten(),
        sample_idx=test_idx,
        filename="outputs/final_ecg_model_comparison.png"
    )
    
    print("[FERTIG] Alle Trainings- und Auswertungsplots wurden erfolgreich in 'outputs/' gespeichert!")

if __name__ == "__main__":
    main()
'''

with open(os.path.join(src_dir, 'train.py'), 'w') as f:
    f.write(train_code)

print("SUCCESS: Skripte wurden aktualisiert. Starte das Training mit Genauigkeits-Verlauf...")

# Training ausführen
os.chdir('/kaggle/working/ppg-to-ecg-deep-learning')
!python src/train.py

SUCCESS: Skripte wurden aktualisiert. Starte das Training mit Genauigkeits-Verlauf...
--- SCHRITT 1: Datensatz laden und vorverarbeiten ---
[SUCCESS] Found 53 valid signal files! Inspecting columns...
[DEBUG] Columns found in bidmc_43_Signals.csv:
['Time [s]', ' RESP', ' PLETH', ' II', ' V', ' AVR']
[COMPLETE] Successfully processed 10600 total window samples!
Training Set: 8480 Samples | Test Set: 2120 Samples

--- SCHRITT 2: Modell 1 trainieren (50 Epochen) ---
I0000 00:00:1793129270.177582    8056 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1793129270.179619    8056 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
Epoch 1/50
I0000 00:00:1793129274.508887    8087 device_compiler.h:196] Compiled cluster u

In [18]:
import os

src_dir = '/kaggle/working/ppg-to-ecg-deep-learning/src'
os.makedirs(src_dir, exist_ok=True)

# 1. dataset.py mit Sliding-Window-Überlappung aktualisieren
dataset_code = '''import os
import glob
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

def load_and_preprocess_data(data_dir="data", window_size=300):
    # Finde alle CSV-Dateien im Datenordner
    csv_files = glob.glob(os.path.join(data_dir, "*.csv"))
    print(f"[SUCCESS] Found {len(csv_files)} signal files! Inspecting columns...")
    
    all_ppg = []
    all_noisy_ecg = []
    all_clean_ecg = []
    
    for file_path in csv_files:
        df = pd.read_csv(file_path)
        
        # Spaltennamen prüfen (typisch für BIDMC: ' PLETH' für PPG, ' II' für ECG)
        cols = df.columns.tolist()
        # Wir suchen flexibel nach PPG und ECG Spalten
        ppg_col = next((c for c in cols if 'PLETH' in c or 'PPG' in c), None)
        ecg_col = next((c for c in cols if 'II' in c or 'ECG' in c), None)
        
        if ppg_col is None or ecg_col is None:
            continue
            
        ppg_signal = df[ppg_col].values
        ecg_signal = df[ecg_col].values
        
        # Normalisierung der Rohsignale (Z-Score)
        ppg_scaler = StandardScaler()
        ecg_scaler = StandardScaler()
        
        ppg_signal = ppg_scaler.fit_transform(ppg_signal.reshape(-1, 1)).flatten()
        ecg_signal = ecg_scaler.fit_transform(ecg_signal.reshape(-1, 1)).flatten()
        
        # Sliding Window mit 75% Überlappung (Schrittweite = window_size // 4 = 75)
        step_size = window_size // 4 
        
        for i in range(0, len(ppg_signal) - window_size, step_size):
            window_ppg = ppg_signal[i : i + window_size]
            window_ecg = ecg_signal[i : i + window_size]
            
            # Künstliches verrauschtes ECG für Modell 2 erzeugen (z. B. durch leichtes Rauschen)
            noise = np.random.normal(0, 0.1, window_size)
            window_noisy_ecg = window_ecg + noise
            
            if len(window_ppg) == window_size:
                all_ppg.append(window_ppg)
                all_noisy_ecg.append(window_noisy_ecg)
                all_clean_ecg.append(window_ecg)
                
    X_ppg = np.array(all_ppg)[..., np.newaxis]        # Shape: (N, 300, 1)
    X_noisy_ecg = np.array(all_noisy_ecg)[..., np.newaxis] # Shape: (N, 300, 1)
    Y_ecg = np.array(all_clean_ecg)[..., np.newaxis]  # Shape: (N, 300, 1)
    
    print(f"[COMPLETE] Successfully processed {X_ppg.shape[0]} total overlapping window samples!")
    return X_ppg, X_noisy_ecg, Y_ecg
'''

with open(os.path.join(src_dir, 'dataset.py'), 'w') as f:
    f.write(dataset_code)

print("SUCCESS: dataset.py wurde mit 75% Fenster-Überlappung aktualisiert!")

# In den Projektordner wechseln und das Training mit den erweiterten Daten starten
os.chdir('/kaggle/working/ppg-to-ecg-deep-learning')
!python src/train.py

SUCCESS: dataset.py wurde mit 75% Fenster-Überlappung aktualisiert!
--- SCHRITT 1: Datensatz laden und vorverarbeiten ---
[SUCCESS] Found 159 signal files! Inspecting columns...
[COMPLETE] Successfully processed 42241 total overlapping window samples!
Training Set: 33792 Samples | Test Set: 8449 Samples

--- SCHRITT 2: Modell 1 trainieren (50 Epochen) ---
I0000 00:00:1793130279.850029   10645 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1793130279.852104   10645 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
Epoch 1/50
I0000 00:00:1793130284.462109   10677 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
845/845 ━━━━━━━━━━━━━━━━━━━━ 12s 